# The coverage of Figure 3's errors over 300 draws (Sect. 4.5)

**In the paper:** Sect. 4.5, "We repeat the analysis over 300 independent noise realizations …".

Fig. 3 quotes the errors of one noise draw. Whether they are the right size, and whether the detection-set fit is
biased, can only be read over many draws. Per draw and parameter, the **pull** and, per draw, the **coverage test**:

$$p = \frac{\hat\theta - \theta_{\rm true}}{\hat\sigma}, \qquad
\Delta\chi^2 = (\hat\theta - \theta_{\rm true})^{\mathsf T}\,\mathbf C^{-1}\,(\hat\theta - \theta_{\rm true}) < \chi^2_{k}(0.683,\ 0.954,\ 0.997).$$

Honest errors give pulls of mean 0 and width 1, and the truth inside the 1, 2, 3σ regions in 68.3, 95.4, 99.7 % of
the draws (binomial scatter of 300 draws: 0.027 at 1σ).

| step | task | calls | paper |
|---|---|---|---|
| 1 | the scenes of Fig. 3, the scan law and the draws (seed `SEED0 + i`) | `sim.Scene.pair`, `scanlaw.load_scan_law`, `scanlaw.load_parallax_factor` | Sect. 4.5; Fig. 3 |
| 2 | one draw through Fig. 3's chain (the one-call forms, each checked in the Fig. 3 notebook); Fig. 3's own draw | `sim.harvest_campaign`, `dyn.clean_detection_set`, `joint.fit_joint` | Fig. 3 |
| 3 | the ensemble, stored | — | Sect. 4.5 |
| 4 | the stored ensemble is the package's: the first draws recomputed | — | — |
| 5 | pulls and coverage of the window-level fit, with one draw's test written out | — | Sect. 4.5 |
| 6 | the detection-set fit: biased, its errors too small | — | Sect. 4.5 |
| 7 | the pull distributions | — | — |

Every scanlib call carries the name of its module (`dyn.wls`, `joint.fit_joint`); the first code cell lists the
modules.

**What it reproduces** (Sect. 4.5). The window-level fit: pull widths within 15 % of one, coverage within 2.2 binomial
standard errors of its nominal values, for all three cases (18 values; the largest deviation is the 1σ coverage
of the position alone in case (a), 0.623 against 0.683). The three cases use the same seeds, and the two (a)
cases have the same windows and differ only in the fitted parameters, so their results are not independent. The detection-set fit: biased beyond four standard errors in position and proper motion
in case (a), its pull widths above 1.1. **Run time.** About 10 s; `FULL = True` recomputes the 900 chains (9 min).

In [ ]:
# the imports: numpy, matplotlib, scipy's chi^2, and the scanlib modules; every scanlib call below carries its
# module's name
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2

from scanlib import instrument as ins      # the instrument: sampling, LSF, the along-scan projection (Eq. 9)
from scanlib import scanlaw                # scan angles, epochs and parallax factors
from scanlib import simulation as sim      # synthetic scenes and their windows
from scanlib import dynamics as dyn        # the fits to the detections (the sinogram fits)
from scanlib import joint                  # the joint fit of all windows, and its helpers

In [ ]:
# the instrument, defined once (Sect. 2.1): 18 samples of 59 mas along scan, a Gaussian LSF of FWHM 100 mas,
# and its per-window margins Delta_1, Delta_2 (App. A.2; derived in sim_margins_lambda_w)
FWHM_MAS, PITCH_MAS, N_SAMPLES = 100.0, 59.0, 18
inst = ins.Instrument.gaussian(fwhm_mas=FWHM_MAS, pitch_mas=PITCH_MAS, n_samples=N_SAMPLES, margins=ins.TOY_MARGINS)

# the switch FULL and the store of the ensemble
FULL = False                               # True: recompute the 900 chains (about 9 min)
STORE = Path("..") / "stored" / "fig3_coverage.npz"

## 1. The scenes and the draws (Sect. 4.5)

Figure 3's scenes and scan law. Draw i of every case uses the seed
`SEED0 + i`, so a draw can be recomputed on its own. The arrays are padded
to five parameters; the four-parameter case leaves the fifth as NaN.

In [ ]:
# the constants of Figure 3's two scenes: (a) a bound companion, (b) an interloper with a parallax difference
# Figure 3's companion at the reference epoch
RHO_MAS = 200.0                            # its separation from the primary [mas]
PA_DEG = 110.0                             # its position angle [deg], from north through east
Q = 0.3                                    # its flux ratio, companion over primary
# (a) bound: relative proper motion [mas/yr]
MU_A = 3.0                                 # in alpha* (east)
MU_D = 9.0                                 # in delta (north)
# (b) interloper: relative proper motion [mas/yr]
MUA_I = 4.0                                # in alpha* (east)
MUD_I = -3.0                               # in delta (north)
DPI_I = 1e3 / 2000.0 - 1e3 / 120.0         # (b) parallax difference [mas]
K_MAX = 2                                  # at most two sources per window

In [ ]:
# the ensemble: the draws per case and their seeds, the three cases, and the levels of the coverage test
N_REAL = 300                               # noise draws per case
SEED0 = 20261106                           # draw i uses seed SEED0 + i
CASES = ("bound", "bound_parallax", "interloper")
N_PAR = 5
LEVELS = (0.683, 0.954, 0.997)             # the 1, 2, 3 sigma regions

In [ ]:
# the two scenes, and the truth of each case (five columns, NaN where a case does not fit d_parallax)
# sim.Scene.pair: the primary at the origin and a companion at (rho, PA) with flux ratio Q, its motion and parallax
scene_a = sim.Scene.pair(RHO_MAS, PA_DEG, Q, pm=(MU_A, MU_D), dpi=0.0)
scene_b = sim.Scene.pair(RHO_MAS, PA_DEG, Q, pm=(MUA_I, MUD_I), dpi=DPI_I)
# positions[1] is the companion's row (dra*, ddec) at the reference epoch; row 0, omitted, is the primary
A0, D0 = scene_a.positions[1]
# TRUTH: one row per case, in the order of CASES; the columns are (dra*, ddec [mas], mu_a, mu_d [mas/yr],
# d_parallax [mas]). Row 0, "bound": four parameters; row 1, "bound_parallax": the same scene, fitted with a
# parallax difference whose truth is zero; row 2, "interloper"
TRUTH = np.array([[A0, D0, MU_A, MU_D, np.nan],
                  [A0, D0, MU_A, MU_D, 0.0],
                  [A0, D0, MUA_I, MUD_I, DPI_I]])

In [ ]:
# the scan law, the parallax factors and the noise
# psi: the scan angle of each transit [rad]; t: its epoch [yr from the reference epoch]
psi, t = scanlaw.load_scan_law(scanlaw.PLEIADES)
fpar = scanlaw.load_parallax_factor(scanlaw.PLEIADES)      # the along-scan parallax factor of each transit
SIGMA_NOISE = sim.PRIMARY_FLUX / 100.0         # S/N 100 for the isolated primary: the noise sigma per sample

## 2. One draw through Fig. 3's chain

The fitted parameters, their quoted errors and covariance, the same for
the detection-set fit that seeds them (`probe_*`), the detection counts and
the cleaning's passes. A draw whose fit has no covariance or
whose optimiser did not settle is recorded with `ok` False.

In [ ]:
# Figure 3's chain on one noise draw of one case (one_draw)
def one_draw(case, seed):
    """Figure 3's chain on one noise draw of one case: the windows, the per-window detections, the detection-set
    fit, and the window-level fit that it seeds.

    Inputs
        case   "bound" (scene a, four parameters), "bound_parallax" (scene a, with d_parallax) or "interloper"
               (scene b, with d_parallax)
        seed   the seed of the noise draw, a whole number
    Returns a table (a dictionary) with ten entries. The arrays have five places, (dra*, ddec [mas], mu_a, mu_d
    [mas/yr], d_parallax [mas]); a case of four parameters leaves the fifth NaN
        "theta"        the parameters of the window-level fit
        "sigma"        their quoted errors
        "cov"          their covariance, 5 x 5
        "probe_theta"  the parameters of the detection-set fit, the seed of the window-level fit
        "probe_sigma"  their quoted errors
        "probe_cov"    their covariance, 5 x 5
        "n_det"        the number of detections
        "n_kept"       the number of detections that the cleaning kept
        "passes"       the number of passes of the cleaning
        "ok"           True when the window-level fit has a covariance and its optimiser settled; when False, the
                       three entries of the window-level fit stay NaN"""
    # step 1, the case: its scene, whether it fits the parallax difference, and its number of parameters
    if case == "interloper":
        scene = scene_b
    else:
        scene = scene_a
    parallax = case != "bound"
    if parallax:
        n_par = 5
    else:
        n_par = 4

    # step 2, the record, empty: NaN in every array, zero in every count
    record = dict(theta=np.full(N_PAR, np.nan), sigma=np.full(N_PAR, np.nan),
               cov=np.full((N_PAR, N_PAR), np.nan), probe_theta=np.full(N_PAR, np.nan),
               probe_sigma=np.full(N_PAR, np.nan), probe_cov=np.full((N_PAR, N_PAR), np.nan),
               n_det=0, n_kept=0, passes=0, ok=False)

    # step 3, the windows and their detections. The generator of the noise: its seed is the draw's seed alone (the
    # case is not in it), so draw i of the three cases has the same stream of random numbers.
    # sim.harvest_campaign: render the windows and reduce each one to its detections (psi, tau, eta, sigma, fpar)
    rng_noise = np.random.default_rng(seed)
    detections = sim.harvest_campaign(scene, (psi, t, fpar), rng_noise, inst, SIGMA_NOISE,
                                 k_max=K_MAX, return_windows=True)
    record["n_det"] = int(detections.eta.size)

    # step 4, the detection-set fit. The parallax factors go to a fit only when the case fits the parallax
    # difference.
    # dyn.clean_detection_set: the map's peak, the detections within 30 mas of it, the Huber fit of the motion to
    # them, then every detection tested against the fitted track and the motion refitted until the kept set is
    # stable. Its fit seeds the joint fit. It returns the fitted parameters, their covariance, a mask of the kept
    # detections, and a record of the cleaning
    if parallax:
        fpar_of_detections = detections.fpar
        fpar_of_transits = fpar
    else:
        fpar_of_detections = None
        fpar_of_transits = None
    probe_theta, probe_cov, keep, info = dyn.clean_detection_set(detections.psi, detections.tau, detections.eta,
                                                          detections.sigma, fpar_of_detections, parallax=parallax)
    record["n_kept"] = int(keep.sum())
    record["passes"] = int(info["iterations"])
    # the stored arrays have five columns: [:n_par] fills the fitted parameters, the rest stays NaN
    record["probe_theta"][:n_par] = probe_theta
    record["probe_sigma"][:n_par] = np.sqrt(np.diag(probe_cov))    # np.diag: the variances (the diagonal)
    record["probe_cov"][:n_par, :n_par] = probe_cov

    # step 5, the window-level fit.
    # joint.fit_joint: the window-level fit, the joint likelihood of all windows (censored ones included), from
    # the seed
    fit = joint.fit_joint(detections.windows, psi, probe_theta, inst, detections.sigma_noise, tau=t,
                    fpar=fpar_of_transits)
    if fit["cov"] is None or not fit["converged"]:
        return record
    record["theta"][:n_par] = fit["theta"]
    record["sigma"][:n_par] = fit["sigma_theta"]
    # the fit's covariance holds the parameters, then the two fluxes: [:n_par, :n_par] keeps the parameters'
    # block, so we omit the rows and columns of the fluxes
    record["cov"][:n_par, :n_par] = fit["cov"][:n_par, :n_par]
    record["ok"] = True
    return record

In [ ]:
# the chain on the first n draws of every case (run)
def run(n):
    """The chain on the first n draws of every case.

    Inputs
        n   the number of draws of each case
    Returns a table (a dictionary) with the ten entries of one_draw(...); each entry is an array whose first index
    is the case (in the order of CASES) and whose second index is the draw: (case, draw, ...)."""
    # step 1, the chains: records[c][i] is the record of draw i of case c; draw i has the seed SEED0 + i
    records = []
    for case in CASES:
        records_of_case = []
        for i in range(n):                 # i: the draw's number
            records_of_case.append(one_draw(case, SEED0 + i))
        records.append(records_of_case)

    # step 2, each entry stacked over the cases and the draws
    stacked = {}
    for key in ("theta", "sigma", "cov", "probe_theta", "probe_sigma", "probe_cov",
                "n_det", "n_kept", "passes", "ok"):
        rows = []
        for records_of_case in records:
            row = []
            for record in records_of_case:
                row.append(record[key])
            rows.append(row)
        stacked[key] = np.array(rows)
    return stacked

In [ ]:
# the chain checked on Figure 3's own draw, seed 14: the detection counts and the fitted proper motion
figure3_draw = one_draw("bound", 14)                # Figure 3's own seed
# theta = (dra*, ddec, mu_a, mu_d): the indices 2 and 3, and the slice [2:4], keep the proper motion
print(f"Figure 3's draw (seed 14): {figure3_draw['n_det']} detections, {figure3_draw['n_kept']} kept, "
      f"mu = ({figure3_draw['theta'][2]:.2f} +/- {figure3_draw['sigma'][2]:.2f}, "
      f"{figure3_draw['theta'][3]:.2f} +/- {figure3_draw['sigma'][3]:.2f}) mas/yr")
# the numbers of the Fig. 3 notebook: 34 detections of which 33 kept; mu = (3.05, 8.89) mas/yr
assert figure3_draw["ok"] and (figure3_draw["n_det"], figure3_draw["n_kept"]) == (34, 33)
assert np.array_equal(np.round(figure3_draw["theta"][2:4], 2), [3.05, 8.89])

## 3. The ensemble

300 draws of each of the three cases, loaded from the store; `FULL = True` recomputes them.

In [ ]:
# the ensemble of 3 x 300 chains: recomputed and stored with FULL, otherwise loaded from the store
# ensemble: a table from an entry's name to its array over (case, draw, ...): the ten entries of one_draw(...), and from
# the store also "truth" and "seeds"
if FULL:
    ensemble = run(N_REAL)
    STORE.parent.mkdir(exist_ok=True)
    # the store: the ten entries, the truth, and the seed of every draw (SEED0 + 0, SEED0 + 1, ...)
    np.savez(STORE, **ensemble, truth=TRUTH, seeds=SEED0 + np.arange(N_REAL))
    print(f"computed and stored {STORE}")
else:
    with np.load(STORE) as stored:
        ensemble = {}
        for key in stored.files:           # every array of the store
            ensemble[key] = stored[key]
    # the store was made with this notebook's seeds and truth (equal_nan: a NaN equals a NaN)
    assert np.array_equal(ensemble["seeds"], SEED0 + np.arange(N_REAL))
    assert np.array_equal(ensemble["truth"], TRUTH, equal_nan=True)
    print(f"loaded {STORE}")
# every chain of every case completed
assert ensemble["ok"].shape == (len(CASES), N_REAL) and ensemble["ok"].all()
print(f"{ensemble['ok'].sum()} of {ensemble['ok'].size} chains complete")

## 4. The stored ensemble is the package's

The first draws of each case, recomputed now, must equal the stored ones bit
for bit.

In [ ]:
# recompute the first 4 draws of each case and require them to equal the store bit for bit
N_CHECK = 4
fresh = run(N_CHECK)
# the arrays are (case, draw, ...): [:, :N_CHECK] keeps every case and the first N_CHECK draws of the store
for key, recomputed in fresh.items():
    assert np.array_equal(recomputed, ensemble[key][:, :N_CHECK], equal_nan=True), \
        f"{key}: the package no longer reproduces the stored ensemble; set FULL = True"
print(f"first {N_CHECK} draws of each case recomputed: identical")

## 5. Pulls and coverage of the window-level fit

The pull of each parameter and the coverage test of the full covariance, and of the position alone (k = 2). First,
one draw's test written out.

In [ ]:
# truth: one draw's coverage test by hand, draw 0 of case (a). [0, 0, :4] keeps case (a), draw 0 and the four
# fitted parameters; the fifth column, d_parallax, is NaN in this case
offset_0 = ensemble["theta"][0, 0, :4] - TRUTH[0, :4]   # fitted minus true: (dra*, ddec [mas], mu_a, mu_d [mas/yr])
covariance_0 = ensemble["cov"][0, 0, :4, :4]                # the quoted covariance of the four parameters
# Delta chi2 = d^T C^-1 d, with d = offset_0 and C = covariance_0: np.linalg.solve(C, d) is C^-1 d, and @ is the
# scalar product
dchi2_0 = float(offset_0 @ np.linalg.solve(covariance_0, offset_0))
# the levels of chi^2 with 4 degrees of freedom that hold 68.3, 95.4 and 99.7 % of the draws (chi2.ppf)
level_texts = []
for probability in LEVELS:
    level_texts.append(f"{chi2.ppf(probability, 4):.2f}")
print(f"draw 0 of (a): fitted minus true {np.round(offset_0, 2)}; Delta chi2 = {dchi2_0:.2f} against "
      + ", ".join(level_texts) + " (1, 2, 3 sigma for 4 parameters)")

In [ ]:
# truth: the pulls and the coverage of one case (summary)
def summary(i, n_par):
    """The pulls and the coverage of the window-level fit of one case, over its draws.

    Inputs
        i       the case's number in CASES: 0, 1 or 2
        n_par   the number of parameters that the case fits: 4, or 5 with d_parallax
    Returns a table (a dictionary) with eight entries
        "n_ok"          the number of draws whose chain completed
        "pulls"         (fitted - true) / quoted error, one row per draw and one column per parameter
        "pull_mean"     the mean of the pulls over the draws, per parameter (0 for an unbiased fit)
        "pull_width"    their standard deviation, per parameter (1 when the quoted errors are right)
        "coverage"      the fraction of the draws whose Delta chi2 of all the parameters is inside the 1, 2 and
                        3 sigma regions (LEVELS)
        "coverage_pos"  the same for the position alone (dra*, ddec)
        "n_det"         the number of detections of every draw
        "n_kept"        the number of detections kept of every draw"""
    # step 1, the draws whose chain completed, and their fitted parameters, quoted errors and covariances;
    # [ok, :n_par] keeps those draws and the first n_par parameters
    ok = ensemble["ok"][i]                                           # a mask: the draws whose chain completed
    theta = ensemble["theta"][i][ok, :n_par]
    sigma = ensemble["sigma"][i][ok, :n_par]
    cov = ensemble["cov"][i][ok, :n_par, :n_par]

    # step 2, the pulls: fitted minus true, over the quoted error
    offsets = theta - TRUTH[i, :n_par]                               # fitted minus true, one row per draw
    pulls = offsets / sigma

    # step 3, Delta chi2 = d^T C^-1 d of every draw, with d the offsets and C the covariance. The einsum sums
    # d[n, i] C^-1[n, i, j] d[n, j] over i and j for each draw n
    dchi2 = np.einsum("ni,nij,nj->n", offsets, np.linalg.inv(cov), offsets)
    # the position alone: [:, :2] keeps the first two parameters of every draw, [:, :2, :2] their 2 x 2 block
    position_offsets = offsets[:, :2]
    position_cov = cov[:, :2, :2]
    dchi2_position = np.einsum("ni,nij,nj->n", position_offsets, np.linalg.inv(position_cov), position_offsets)

    # step 4, the coverage: the fraction of the draws with Delta chi2 below the level of chi^2 (n_par degrees of
    # freedom; 2 for the position) that holds the probability 0.683, 0.954 or 0.997
    coverage = []
    coverage_pos = []
    for probability in LEVELS:
        coverage.append(np.mean(dchi2 < chi2.ppf(probability, n_par)))
        coverage_pos.append(np.mean(dchi2_position < chi2.ppf(probability, 2)))
    # mean(0), std(0): over the draws (axis 0), so one number per parameter
    return dict(n_ok=int(ok.sum()), pulls=pulls, pull_mean=pulls.mean(0), pull_width=pulls.std(0),
                coverage=np.array(coverage),
                coverage_pos=np.array(coverage_pos),
                n_det=ensemble["n_det"][i][ok], n_kept=ensemble["n_kept"][i][ok])

In [ ]:
# truth: the three cases summarized and printed; the asserts say that the quoted errors are honest (Sect. 4.5)
expected = np.array(LEVELS)
worst_deviation = 0.0                      # the largest |coverage - expected| of the 18 values, in binomial sigmas
TAGS = ("(a) four parameters", "(a) with d_parallax, the null test", "(b) interloper")   # one per case of CASES
# S: a table from a case's name to its summary (the table that summary(...) returns)
S = {}
for i, (case, tag) in enumerate(zip(CASES, TAGS)):     # i: the case's number; case: its name; tag: its label
    if case == "bound":
        n_par = 4
    else:
        n_par = 5
    case_summary = summary(i, n_par)
    S[case] = case_summary
    print(f"{tag}: {case_summary['n_ok']} draws, {case_summary['n_det'].mean():.1f} detections of which "
          f"{case_summary['n_kept'].mean():.1f} kept on average")
    print(f"    pull mean  {np.round(case_summary['pull_mean'], 2)}")
    print(f"    pull width {np.round(case_summary['pull_width'], 2)}   (1 if the quoted errors are right)")
    print(f"    coverage   {np.round(case_summary['coverage'], 3)} all parameters, "
          f"{np.round(case_summary['coverage_pos'], 3)} position (expected {expected})")
    # honest errors: widths within 15 % of one, no pull mean beyond 0.2,
    # coverage within 3 binomial sigma of nominal
    assert np.all(np.abs(case_summary["pull_width"] - 1) < 0.15) and np.all(np.abs(case_summary["pull_mean"]) < 0.2)
    # the binomial sigma of a fraction p over n draws is sqrt(p (1 - p) / n)
    binomial_sigma = np.sqrt(expected * (1 - expected) / case_summary["n_ok"])
    tolerance = 3 * binomial_sigma
    assert np.all(np.abs(case_summary["coverage"] - expected) < tolerance)
    assert np.all(np.abs(case_summary["coverage_pos"] - expected) < tolerance)
    # the largest deviation of this case's six values (all parameters, then the position), in binomial sigmas
    deviation_all = np.abs(case_summary["coverage"] - expected) / binomial_sigma
    deviation_position = np.abs(case_summary["coverage_pos"] - expected) / binomial_sigma
    worst_deviation = max(worst_deviation, deviation_all.max(), deviation_position.max())
# the introduction's "within 2.2 binomial standard errors"
assert round(float(worst_deviation), 1) == 2.2
print("the quoted errors of all three fits are honest")

## 6. The detection-set fit (Sect. 4.5)

The same reading for the fit that seeds the window-level fit: its bias (the
mean of fitted minus true) against its standard error, and its pull width.

In [ ]:
# truth: the detection-set fit against the window-level fit: the mean offset from the truth over its standard error,
# and the pull width
for i, case in enumerate(CASES):           # i: the case's number; case: its name
    if case == "bound":
        n_par = 4
    else:
        n_par = 5
    ok = ensemble["ok"][i]                        # a mask: the draws whose chain completed
    # the two fits: its printed name, and the prefix of its entries in ensemble ("probe_theta" or "theta", ...)
    for tag, key in (("detection set", "probe_"), ("window level", "")):
        offsets = ensemble[key + "theta"][i][ok, :n_par] - TRUTH[i, :n_par]     # fitted minus true, one row per draw
        standard_error = offsets.std(0) / np.sqrt(len(offsets))    # of the mean offset, per parameter
        width = (offsets / ensemble[key + "sigma"][i][ok, :n_par]).std(0)       # the pull width, per parameter
        bias_over_error = offsets.mean(0) / standard_error         # the mean offset in standard errors, per parameter
        print(f"{case:15s} {tag:13s} bias/se {np.round(bias_over_error, 1)}  pull width {np.round(width, 2)}")
        if tag == "window level":
            # unbiased: no mean offset beyond three standard errors
            assert np.all(np.abs(offsets.mean(0)) < 3 * standard_error)
        else:
            # [:4] keeps the position and the proper motion, so we omit d_parallax; [:n_wide] keeps the widths
            # that must exceed 1.1
            # floor: the least bias, in standard errors, that the assert requires of the four parameters
            if case == "interloper":
                floor = np.array([4, 4, 2, 2])
                n_wide = 2                 # the interloper's proper-motion pull widths are 1.09, 1.10
            else:
                floor = 4
                n_wide = 4
            assert np.all(np.abs(offsets.mean(0)[:4]) > floor * standard_error[:4]) and np.all(width[:n_wide] > 1.1)
print("the detection-set fit is biased and its errors are too small; the window-level fit is neither")

## 7. The pull distributions

Per case and parameter, the pulls of the window-level fit against the standard normal.

In [ ]:
# the figure: the pulls of the window-level fit, per case and parameter, against the standard normal
names = [r"$\Delta\alpha^*_0$", r"$\Delta\delta_0$", r"$\mu_{\alpha^*}$", r"$\mu_\delta$",
         r"$\Delta\varpi$"]                # the label of each parameter, in the order of theta
grid = np.linspace(-4, 4, 200)             # the pulls at which the standard normal is drawn
PAPER_STYLE = {"font.size": 10.0, "xtick.labelsize": 9.0, "ytick.labelsize": 9.0, "axes.linewidth": 0.5,
               "xtick.direction": "in", "ytick.direction": "in"}
with plt.rc_context(PAPER_STYLE):                                         # the paper's style, this plot only
    # one row of panels per case, one column per parameter
    fig, axes = plt.subplots(len(CASES), N_PAR, figsize=(10.0, 5.6), sharex=True, sharey=True)
    for row, case in zip(axes, CASES):
        pulls = S[case]["pulls"]           # one row per draw, one column per fitted parameter
        for k, ax in enumerate(row):       # k: the parameter's number
            # a case of four parameters has no fifth column: its fifth panel is left empty
            if k >= pulls.shape[1]:
                ax.axis("off")
                continue
            ax.hist(pulls[:, k], bins=np.linspace(-4, 4, 25), density=True, color="0.7")
            # the standard normal density
            ax.plot(grid, np.exp(-0.5 * grid ** 2) / np.sqrt(2 * np.pi), "k-", lw=0.8)
            ax.text(0.04, 0.95, f"{case}\n{names[k]}", transform=ax.transAxes, va="top", fontsize=7)
    for ax in axes[-1]:                    # axes[-1]: the bottom row
        ax.set_xlabel("pull")
    plt.tight_layout()
    plt.show()